# PrefixLM with ModernBERT

### Training task: P(Seq1 | Seq2) using a packed sequence `[CLS] seq2 [SEP] seq1 [SEP]`
### with a custom PrefixLM attention mask passed as a dict to ModernBERT.

# Imports

In [8]:

import torch
import torch.nn as nn
import math
from torch.nn import CrossEntropyLoss


from transformers import (
    ModernBertConfig,
    ModernBertForMaskedLM,
    ModernBertModel
)

from datasets import load_from_disk
 
# Your gLM imports
from gLM.dataset import Uniref90ArrowDatasetForFASTA
from gLM.tokenizers import PhyloTokenizerLoader

from gLM.sequences.pairwise_align import align_pair, percent_identity
from gLM.models import ProteinBertModel
from gLM.collator import create_mlm_collator, PhyloCollator
from gLM.train_utils import PhyloTrainer


# Load tokenizer

In [9]:
tokenizer = PhyloTokenizerLoader("./phylo_char_tokenizer_with_bos")
print("bos_token:", tokenizer.bos_token)
print("bos_token_id:", tokenizer.bos_token_id)
print("pad_token_id:", tokenizer.pad_token_id)
print("sep_token_id:", tokenizer.sep_token_id)
print("cls_token_id:", tokenizer.cls_token_id)
print("vocab_size:", tokenizer.vocab_size)

bos_token: [BOS]
bos_token_id: 27
pad_token_id: 0
sep_token_id: 3
cls_token_id: 2
vocab_size: 27


# Load Dataset

In [10]:
dataset_path = "/gpfs/data/brandeslab/Data/uniref/uniref90_clusters_arrow/train"
fasta_path = "/gpfs/data/brandeslab/Data/uniref/uniref100.fasta"
idx_db_path = "/gpfs/data/brandeslab/User/as12267/uniref100.idx"
 
ds = Uniref90ArrowDatasetForFASTA(
    dataset_path=dataset_path,
    training_type="phylo_encoder_decoder",
    fasta_path=fasta_path,
    idx_db_path=idx_db_path,
)
 
batch_raw = [ds[i] for i in range(4)]
for i, (s1, s2) in enumerate(batch_raw):
    print(f"\nExample {i}")
    print("s1:", s1[:120])
    print("s2:", s2[:120])
    print("len(s1):", len(s1), "len(s2):", len(s2))


Example 0
s1: MSKKEEDFIENLNLSRDISSEFAIAEKIIRTSAKGRKYIDITIVDRTGQMDGRMFPHPIEVDSVHDSIKLGSVCRIMGRISEFPSDSGKFNMVINVLTELDDDEYQLEDFVMASENNTDD
s2: MSKKEEDFVENLNLSRDISSEFAIAEKIIRTSAKGRKYIDITLVDRTGQMDGRMFPHPIEVDSVHASIKLGSVCKIIGRISEFPSDSGKFNMVINVLTELDDDEYQLEDFVMASENNTDD
len(s1): 300 len(s2): 300

Example 1
s1: MKVRASIKRICNNCKIIKRHGVNRVICINPKHKQRQG
s2: MKVRASIKRICKDCKIIKRHGVNRVICINPKHKQRQG
len(s1): 37 len(s2): 37

Example 2
s1: MVEPDDETPRVGGTDDRSIYERALGEEFTALHPKVQERFGFTSADGVACIGRGTMEYVRNGGPHLLPFLWFGATHNTMFPEENTAVPFTIRNYAYEDAFGRETVTWLRRFDMPRRRRFDA
s2: MVELDDETPRVGGTDGRSIYERALGEEFTALHPKIQERFGFTSADGVACIGRGTMEYVRNGGPHLLPFLWFGATHNTMFPEQNTAVPFTIRNYAYEDAFGRETVTWLRRFDMPRRRRFDA
len(s1): 238 len(s2): 238

Example 3
s1: MEYDFIYDRNTLQFSIKLAAEHEVLGRFLLDEFGQQAEAYQAIIQQLSEQNAHQPMQYQGKEFLLEVDEGEVTLSHNTLFYSAEHPTTEQQQKLEDDDLQLDEQGMTMQCGLEDLLKLLR
s2: MEYDFIYDRNTLQFSIKLAAEHEVLGRFLLDEFGQQAEAYQAIIQQLSEQNAHQHMQFQGKEFLLEVEEGEVTLSHNTLFYSAEHPTTEQQQKLEDDDLHLDEQGMTMQCGLEDLLKLLR
len(s1): 128 len(s2): 128


# PrefixLM Attention Mask
#
#           prefix    suffix
###  prefix  [  full       0    ]   ← bidirectional, no peeking at suffix
###  suffix  [  full    causal   ]  ← sees all prefix + causal within suffix
###  
#
###  Returned as a dict `{"full_attention": mask, "sliding_attention": mask}`.
###  ModernBertModel.forward (line 465) checks `isinstance(attention_mask, dict)`
###  and if so, bypasses all internal mask creation and uses your dict directly.


In [25]:
def build_prefixlm_4d_mask(prefix_lengths, input_ids, pad_token_id, dtype):
    """
    Build a 4D FLOAT attention mask in the format ModernBERT layers expect.
    0.0 = attend, -inf = don't attend.
    Shape: (batch, 1, seq_len, seq_len)
    """
    B, T = input_ids.shape
    device = input_ids.device

    row = torch.arange(T, device=device)
    col = torch.arange(T, device=device)
    P = prefix_lengths[:, None].to(device)  # (B, 1)

    is_prefix_row = (row[None, :, None] < P[:, :, None])
    is_prefix_col = (col[None, None, :] < P[:, None, :])

    # PrefixLM pattern (bool: True = attend)
    prefix_bidir = is_prefix_row & is_prefix_col
    suffix_to_prefix = (~is_prefix_row) & is_prefix_col
    suffix_causal = (~is_prefix_row) & (~is_prefix_col) & (row[None, :, None] >= col[None, None, :])
    bool_mask = prefix_bidir | suffix_to_prefix | suffix_causal

    # Apply padding: don't attend to/from pad tokens
    padding_mask = (input_ids != pad_token_id)  # (B, T)
    pad_2d = padding_mask[:, :, None] & padding_mask[:, None, :]  # (B, T, T)
    bool_mask = bool_mask & pad_2d

    # Convert to float: True → 0.0, False → -inf
    float_mask = torch.where(
        bool_mask.unsqueeze(1),  # (B, 1, T, T)
        torch.tensor(0.0, dtype=dtype, device=device),
        torch.tensor(torch.finfo(dtype).min, dtype=dtype, device=device),
    )
    return float_mask


In [26]:
def visualize_mask(prefix_len, total_len):
    P = torch.tensor([prefix_len])
    mask = build_prefixlm_mask(P, total_len, device=torch.device("cpu"))
    m = mask[0, 0].int()
    header = "     " + " ".join([f"{j:>2}" for j in range(total_len)])
    print(header)
    for i in range(total_len):
        label = "P" if i < prefix_len else "S"
        row = " ".join(["■ " if m[i, j] else "· " for j in range(total_len)])
        print(f"  {label}{i:>2} {row}")
 
print("PrefixLM mask (prefix=5, total=9):")
visualize_mask(prefix_len=5, total_len=9)

PrefixLM mask (prefix=5, total=9):
      0  1  2  3  4  5  6  7  8
  P 0 ■  ■  ■  ■  ■  ·  ·  ·  · 
  P 1 ■  ■  ■  ■  ■  ·  ·  ·  · 
  P 2 ■  ■  ■  ■  ■  ·  ·  ·  · 
  P 3 ■  ■  ■  ■  ■  ·  ·  ·  · 
  P 4 ■  ■  ■  ■  ■  ·  ·  ·  · 
  S 5 ■  ■  ■  ■  ■  ■  ·  ·  · 
  S 6 ■  ■  ■  ■  ■  ■  ■  ·  · 
  S 7 ■  ■  ■  ■  ■  ■  ■  ■  · 
  S 8 ■  ■  ■  ■  ■  ■  ■  ■  ■ 


# PrefixLM Collator
#
### Your dataset returns `(seq1, seq2)` pairs. This collator packs them as:
### `[CLS] seq2 [SEP] seq1 [SEP]` and builds the PrefixLM attention mask dict.


In [24]:
class PrefixLMCollator:
    def __init__(self, tokenizer, max_seq_len: int):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __call__(self, batch):
        s1s, s2s = zip(*batch)

        enc_s2 = self.tokenizer(list(s2s), add_special_tokens=False)["input_ids"]
        enc_s1 = self.tokenizer(list(s1s), add_special_tokens=False)["input_ids"]

        cls_id = self.tokenizer.cls_token_id
        sep_id = self.tokenizer.sep_token_id
        pad_id = self.tokenizer.pad_token_id

        all_input_ids = []
        all_prefix_lengths = []

        for s2_ids, s1_ids in zip(enc_s2, enc_s1):
            packed = [cls_id] + s2_ids + [sep_id] + s1_ids + [sep_id]
            if len(packed) > self.max_seq_len:
                overflow = len(packed) - self.max_seq_len
                s1_ids = s1_ids[: len(s1_ids) - overflow]
                packed = [cls_id] + s2_ids + [sep_id] + s1_ids + [sep_id]

            all_input_ids.append(packed)
            all_prefix_lengths.append(1 + len(s2_ids) + 1)

        max_len = max(len(ids) for ids in all_input_ids)

        padded_input_ids = []
        labels_list = []

        for input_ids, prefix_len in zip(all_input_ids, all_prefix_lengths):
            pad_len = max_len - len(input_ids)
            labels = list(input_ids)
            for i in range(prefix_len):
                labels[i] = -100
            labels[-1] = -100

            padded_input_ids.append(input_ids + [pad_id] * pad_len)
            labels_list.append(labels + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "labels": torch.tensor(labels_list, dtype=torch.long),
            "prefix_lengths": torch.tensor(all_prefix_lengths, dtype=torch.long),
        }

# Test collator with real data

In [15]:

collator = PrefixLMCollator(tokenizer=tokenizer, max_seq_len=1024)
batch_out = collator(batch_raw)
 
print("=== Collator output ===")
print(f"input_ids shape:      {batch_out['input_ids'].shape}")
print(f"labels shape:         {batch_out['labels'].shape}")
print(f"prefix_lengths:       {batch_out['prefix_lengths'].tolist()}")
print(f"attention_mask keys:  {list(batch_out['attention_mask'].keys())}")
print(f"full_attn mask shape: {batch_out['attention_mask']['full_attention'].shape}")
 
for i in range(len(batch_raw)):
    plen = batch_out["prefix_lengths"][i].item()
    assert (batch_out["labels"][i, :plen] == -100).all(), f"Sample {i}: prefix labels must be -100"
    seq_len = (batch_out["input_ids"][i] != tokenizer.pad_token_id).sum().item()
    suffix_labels = batch_out["labels"][i, plen:seq_len - 1]
    assert (suffix_labels != -100).all(), f"Sample {i}: suffix labels must be real tokens"
    print(f"  Sample {i}: prefix_len={plen}, total_len={seq_len}, suffix_tokens={seq_len - plen}")
 
print("✓ All label checks passed")

=== Collator output ===
input_ids shape:      torch.Size([4, 603])
labels shape:         torch.Size([4, 603])
prefix_lengths:       [302, 39, 240, 130]
attention_mask keys:  ['full_attention', 'sliding_attention']
full_attn mask shape: torch.Size([4, 1, 603, 603])
  Sample 0: prefix_len=302, total_len=603, suffix_tokens=301
  Sample 1: prefix_len=39, total_len=77, suffix_tokens=38
  Sample 2: prefix_len=240, total_len=479, suffix_tokens=239
  Sample 3: prefix_len=130, total_len=259, suffix_tokens=129
✓ All label checks passed


# Build model
#
### Using your ProteinBertModel class. **Must use `attn_implementation="sdpa"`**
### because SDPA accepts arbitrary 4D boolean masks, flash_attention_2 does not.


In [16]:
builder = ProteinBertModel(
    vocab_size=tokenizer.vocab_size,
    tokenizer=tokenizer,
    attn_implementation="sdpa",  # REQUIRED for custom 4D PrefixLM mask
)
model = builder.build()
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")


Using sdpa attention
Model params: 113,876,763


# Custom forward with autoregressive loss


In [27]:
def prefixlm_forward(model, batch, device):
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    prefix_lengths = batch["prefix_lengths"].to(device)

    encoder = model.model  # ModernBertModel
    B, T = input_ids.shape

    # Build 4D float mask in the format _update_attention_mask would produce
    mask_4d = build_prefixlm_4d_mask(
        prefix_lengths, input_ids, model.config.pad_token_id, encoder.dtype
    )

    # Position IDs
    position_ids = torch.arange(T, device=device).unsqueeze(0)

    # Embeddings
    hidden_states = encoder.embeddings(input_ids=input_ids)

    # Run each encoder layer — pass our mask as BOTH attention_mask and sliding_window_mask
    # This makes every layer (global and local) use the full PrefixLM pattern
    for layer in encoder.layers:
        layer_outputs = layer(
            hidden_states,
            attention_mask=mask_4d,
            sliding_window_mask=mask_4d,  # override local attn with same mask
            position_ids=position_ids,
        )
        hidden_states = layer_outputs[0]

    # Final norm
    hidden_states = encoder.final_norm(hidden_states)

    # LM head (reuse ModernBertForMaskedLM's head + decoder)
    logits = model.decoder(model.head(hidden_states))

    # Autoregressive shifted loss on suffix only
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    loss = CrossEntropyLoss(ignore_index=-100)(
        shift_logits.view(-1, model.config.vocab_size),
        shift_labels.view(-1),
    )
    return loss, logits

# Forward + backward pass


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.train()
 
loss, logits = prefixlm_forward(model, batch_out, device)
print(f"Loss: {loss.item():.4f}")
print(f"Logits shape: {logits.shape}")
 
loss.backward()
 
grad_count = sum(1 for _, p in model.named_parameters() if p.grad is not None and p.grad.abs().sum() > 0)
total_params = sum(1 for _ in model.parameters())
print(f"Parameters with nonzero gradients: {grad_count} / {total_params}")
 
import math
print(f"Expected random loss: ~{math.log(tokenizer.vocab_size):.2f}")
print(f"Actual loss:          {loss.item():.2f}")
print("✓ Forward + backward pass successful")
 
model.zero_grad()

/gpfs/data/brandeslab/User/as12267/.conda/envs/huggingface_bert_cu126/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Loss: 3.4757
Logits shape: torch.Size([4, 603, 27])
Parameters with nonzero gradients: 77 / 77
Expected random loss: ~3.30
Actual loss:          3.48
✓ Forward + backward pass successful


# Sanity checks: no leakage + prefix conditions suffix

In [30]:
def _run_encoder(model, input_ids, prefix_lengths, device):
    """Helper: run encoder bypassing model.model.forward()"""
    encoder = model.model
    B, T = input_ids.shape
    mask_4d = build_prefixlm_4d_mask(prefix_lengths, input_ids, model.config.pad_token_id, encoder.dtype)
    position_ids = torch.arange(T, device=device).unsqueeze(0)
    hidden_states = encoder.embeddings(input_ids=input_ids)
    for layer in encoder.layers:
        layer_outputs = layer(
            hidden_states,
            attention_mask=mask_4d,
            sliding_window_mask=mask_4d,
            position_ids=position_ids,
        )
        hidden_states = layer_outputs[0]
    return encoder.final_norm(hidden_states)


## Check 1: Corrupting a suffix token should NOT change prefix hidden states


In [31]:
def check_no_leakage(model, batch_out, device):
    model.eval()
    input_ids = batch_out["input_ids"].to(device)
    prefix_lengths = batch_out["prefix_lengths"].to(device)
    prefix_len = prefix_lengths[0].item()

    with torch.no_grad():
        h1 = _run_encoder(model, input_ids, prefix_lengths, device)
        prefix_hidden_1 = h1[0, :prefix_len].clone()

    corrupted_ids = input_ids.clone()
    suffix_pos = prefix_len + 1
    if suffix_pos < input_ids.shape[1]:
        corrupted_ids[0, suffix_pos] = (corrupted_ids[0, suffix_pos] + 1) % model.config.vocab_size
        with torch.no_grad():
            h2 = _run_encoder(model, corrupted_ids, prefix_lengths, device)
            prefix_hidden_2 = h2[0, :prefix_len].clone()

        diff = (prefix_hidden_1 - prefix_hidden_2).abs().max().item()
        print(f"Max diff in prefix hidden states after corrupting suffix: {diff:.10f}")
        assert diff < 1e-5, f"LEAKAGE DETECTED! diff={diff}"
        print("✓ No information leakage from suffix to prefix")
    model.train()


## Check 2: Corrupting a prefix token SHOULD change suffix hidden states


In [32]:
def check_prefix_conditions_suffix(model, batch_out, device):
    model.eval()
    input_ids = batch_out["input_ids"].to(device)
    prefix_lengths = batch_out["prefix_lengths"].to(device)
    prefix_len = prefix_lengths[0].item()

    with torch.no_grad():
        h1 = _run_encoder(model, input_ids, prefix_lengths, device)
        suffix_hidden_1 = h1[0, prefix_len:].clone()

    corrupted_ids = input_ids.clone()
    corrupted_ids[0, 1] = (corrupted_ids[0, 1] + 1) % model.config.vocab_size
    with torch.no_grad():
        h2 = _run_encoder(model, corrupted_ids, prefix_lengths, device)
        suffix_hidden_2 = h2[0, prefix_len:].clone()

    diff = (suffix_hidden_1 - suffix_hidden_2).abs().max().item()
    print(f"Max diff in suffix hidden states after corrupting prefix: {diff:.6f}")
    assert diff > 1e-3, f"Prefix doesn't affect suffix! diff={diff}"
    print("✓ Prefix correctly conditions the suffix")
    model.train()

In [33]:
check_no_leakage(model, batch_out, device)


Max diff in prefix hidden states after corrupting suffix: 0.0000000000
✓ No information leakage from suffix to prefix


In [34]:
check_prefix_conditions_suffix(model, batch_out, device)

Max diff in suffix hidden states after corrupting prefix: 0.003196
✓ Prefix correctly conditions the suffix


# Training with your PhyloTrainer
### Override `compute_loss` to use autoregressive loss instead of MLM loss.



In [19]:
class PrefixLMTrainer(PhyloTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        input_ids = inputs["input_ids"]
        labels = inputs["labels"]
        attention_mask = inputs["attention_mask"]  # already a dict from collator
 
        outputs = model.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state
        logits = model.decoder(model.head(hidden_states))
 
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
 
        loss_fct = CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(
            shift_logits.view(-1, model.config.vocab_size),
            shift_labels.view(-1),
        )
        return (loss, {"logits": logits}) if return_outputs else loss
 

# Launch training


In [ ]:
from transformers import TrainingArguments
 
collator = PrefixLMCollator(tokenizer=tokenizer, max_seq_len=4096)
 
training_args = TrainingArguments(
    output_dir="./prefixlm_protein_test",
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=50,  # short run for testing
    logging_steps=5,
    bf16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,  # IMPORTANT: keeps prefix_lengths and dict attention_mask
)
 
trainer = PrefixLMTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=ds,
)
 
trainer.train()

In [20]:
import transformers
print(transformers.__version__)

4.57.1


In [21]:
import inspect
from transformers import ModernBertModel
# Print the forward method source to see how it handles attention_mask
source = inspect.getsource(ModernBertModel.forward)
print(source)

    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        sliding_window_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        indices: Optional[torch.Tensor] = None,
        cu_seqlens: Optional[torch.Tensor] = None,
        max_seqlen: Optional[int] = None,
        batch_size: Optional[int] = None,
        seq_len: Optional[int] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[tuple[torch.Tensor, ...], BaseModelOutput]:
        r"""
        sliding_window_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):
            Mask to avoid performing attention on padding or far-away tokens. In ModernBert, only every few layers
            pe

In [22]:
import inspect

# 1. What does _update_attention_mask produce?
print(inspect.getsource(model.model._update_attention_mask))

    def _update_attention_mask(self, attention_mask: torch.Tensor, output_attentions: bool) -> torch.Tensor:
        if output_attentions:
            if self.config._attn_implementation == "sdpa":
                logger.warning_once(
                    "Outputting attentions is only supported with the 'eager' attention implementation, "
                    'not with "sdpa". Falling back to `attn_implementation="eager"`.'
                )
                self.config._attn_implementation = "eager"
            elif self.config._attn_implementation != "eager":
                logger.warning_once(
                    "Outputting attentions is only supported with the eager attention implementation, "
                    f'not with {self.config._attn_implementation}. Consider setting `attn_implementation="eager"`.'
                    " Setting `output_attentions=False`."
                )

        global_attention_mask = _prepare_4d_attention_mask(attention_mask, self.dtype)

        # Cr

In [23]:
# 2. What does a single encoder layer expect?
print(inspect.getsource(model.model.layers[0].forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        sliding_window_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        cu_seqlens: Optional[torch.Tensor] = None,
        max_seqlen: Optional[int] = None,
        output_attentions: Optional[bool] = False,
    ) -> torch.Tensor:
        attn_outputs = self.attn(
            self.attn_norm(hidden_states),
            attention_mask=attention_mask,
            sliding_window_mask=sliding_window_mask,
            position_ids=position_ids,
            cu_seqlens=cu_seqlens,
            max_seqlen=max_seqlen,
            output_attentions=output_attentions,
        )
        hidden_states = hidden_states + attn_outputs[0]
        mlp_output = (
            self.compiled_mlp(hidden_states)
            if self.config.reference_compile
            else self.mlp(self.mlp_norm(hidden_states))
        )
        hidden_st